In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
input_dim = 784
hidden_dim = 400
latent_dim = 20
batch_size = 128
epochs = 50
learning_rate = 1e-3

# MNIST Dataset
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data",
                               train=True,
                               transform=transform,
                               download=True)

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True)


# VAE Model
class VAE(nn.Module):

    def __init__(self):
        super(VAE, self).__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    # Encoder
    def encode(self, x):
        h = self.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    # Reparameterization Trick
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mu + epsilon * std
        return z

    # Decoder
    def decode(self, z):
        h = self.relu(self.fc2(z))
        x_recon = self.sigmoid(self.fc3(h))
        return x_recon

    # Forward Pass
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)
        return reconstruction, mu, logvar


# Initialize model
model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Reconstruction Loss
reconstruction_loss_fn = nn.BCELoss(reduction='sum')


# Training Loop
for epoch in range(epochs):

    total_loss = 0
    total_recon_loss = 0
    total_kl_loss = 0

    for images, _ in train_loader:

        images = images.view(-1, 784).to(device)

        # Forward pass
        reconstruction, mu, logvar = model(images)

        # Reconstruction loss
        recon_loss = reconstruction_loss_fn(reconstruction, images)

        # KL Divergence Loss
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        # Total VAE loss
        loss = recon_loss + kl_loss

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_kl_loss += kl_loss.item()

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Total Loss: {total_loss:.2f}")
    print(f"Reconstruction Loss: {total_recon_loss:.2f}")
    print(f"KL Divergence Loss: {total_kl_loss:.2f}")
    print("-"*40)

Epoch [1/50]
Total Loss: 9855256.86
Reconstruction Loss: 8925682.52
KL Divergence Loss: 929574.35
----------------------------------------
Epoch [2/50]
Total Loss: 7275861.17
Reconstruction Loss: 5928605.48
KL Divergence Loss: 1347255.69
----------------------------------------
Epoch [3/50]
Total Loss: 6867310.61
Reconstruction Loss: 5427942.49
KL Divergence Loss: 1439368.12
----------------------------------------
Epoch [4/50]
Total Loss: 6692946.13
Reconstruction Loss: 5219552.26
KL Divergence Loss: 1473393.87
----------------------------------------
Epoch [5/50]
Total Loss: 6593429.66
Reconstruction Loss: 5103066.00
KL Divergence Loss: 1490363.65
----------------------------------------
Epoch [6/50]
Total Loss: 6520855.60
Reconstruction Loss: 5022085.94
KL Divergence Loss: 1498769.66
----------------------------------------
Epoch [7/50]
Total Loss: 6468735.93
Reconstruction Loss: 4964803.34
KL Divergence Loss: 1503932.59
----------------------------------------
Epoch [8/50]
Total Lo